# 🧫 Bioreactor Excel Preprocessor
**Input:** `Bioreactor_현황판.xlsx` (with cell comments for user names)  
**Output:** `gantt_import.csv` → upload to Bioreactor App Admin Section 3

---
### Instructions
1. Set `INPUT_FILE` to your Excel file path
2. Set `OUTPUT_FILE` to where you want the CSV saved
3. Run all cells (Cell → Run All)

In [1]:
# ── SETTINGS — edit these paths ───────────────────────────────────────────────
INPUT_FILE  = 'C:/Users/chan85.kim/Desktop/Jupyter Lab/2026/배양기 운용현황판_streamlit/260514_Bioreactor_현황판.xlsx'   # path to your Excel file
OUTPUT_FILE = 'C:/Users/chan85.kim/Desktop/Jupyter Lab/2026/배양기 운용현황판_streamlit/extracted.csv'         # output CSV file name
SHEET_INDEX = 0                          # 0 = first sheet (배양기 일정)

In [2]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'openpyxl', '-q'])
print('✅ openpyxl ready')

✅ openpyxl ready


In [3]:
import openpyxl
from datetime import date
import csv
from collections import defaultdict

In [4]:
# ── Constants ──────────────────────────────────────────────────────────────────
VALID_FLOORS = {'6층', '5층', '7층'}
SKIP_VALUES = {'-', 'w', ''}
CONT_PREFIX = 'D'

MONTH_MAP = {
    'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,
    'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12,
    'January':1,'February':2,'March':3,'April':4,'June':6,
    'July':7,'August':8,'September':9,'October':10,
    'November':11,'December':12
}

# Valid reactor IDs — must match utils.py INVENTORY_CONFIGS
VALID_IDS = set()
_INV = [
    ('5L',   1,  8, False, '6층'), ('5L',  1,  3, True,  '6층'),
    ('2L',   1, 10, False, '6층'), ('2L', 17, 22, False, '6층'),
    ('2L',  87, 92, False, '6층'), ('2L', 11, 16, False, '6층'),
    ('2L',  23, 46, False, '6층'), ('2L', 47, 74, False, '6층'),
    ('2L',  75, 86, False, '6층'), ('2L',  1, 24, True,  '6층'),
    ('Wave', 3,  5, False, '6층'),
    ('Wave', 1,  2, False, '5층'), ('5L',   1,  4, False, '5층'),
    ('10L',  1,  2, False, '5층'), ('50L',  1,  2, False, '5층'),
    ('50L',  3,  4, False, '5층'),
    ('200L', 1,  5, False, '5층'),
    ('2L',   1, 12, False, '7층'), ('Wave', 6,  7, False, '7층'),
    ('10L',  1,  2, False, '7층'), ('50L',  5,  6, False, '7층'),
    ('200L', 6,  6, False, '7층'),
]
for _sc, _s, _e, _isp, _room in _INV:
    for _i in range(_s, _e + 1):
        if _sc == '5L' and _isp: _n = f'P{_i:02d}'
        elif _isp: _n = f'P{_i}'
        else: _n = f'{_i:02d}'
        VALID_IDS.add(f'{_room}-{_sc}-{_n}')

print(f'✅ {len(VALID_IDS)} valid reactor IDs loaded')

✅ 166 valid reactor IDs loaded


In [5]:
# ── Helper functions ───────────────────────────────────────────────────────────
def normalise(floor, vessel):
    """Normalise vessel ID to match app format."""
    v = vessel.strip().replace(' ', '')
    if v.startswith('W-'):
        v = 'Wave-' + v[2:]
    parts = v.rsplit('-', 1)
    if len(parts) == 2:
        prefix, num = parts
        if num.startswith('P') and num[1:].isdigit():
            n = int(num[1:])
            num = f'P{n:02d}' if '5L' in prefix else f'P{n}'
        elif num.isdigit():
            num = f'{int(num):02d}'
        v = f'{prefix}-{num}'
    return f'{floor}-{v}'


def extract_user(cell):
    """
    Extract department + SME from cell comment (second line onwards).
    Comment format: '나준호(Joonho Na):\n배양 김다혜'
    Returns: '배양 김다혜' (not the creator name)
    """
    if not cell.comment:
        return 'Unknown'
    text = cell.comment.text.strip()
    if not text:
        return 'Unknown'
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if len(lines) >= 2:
        return ' '.join(lines[1:]) # dept + SME from second line onwards
    return 'Unknown'


def build_date_map(ws):
    """Build {col_index: date} from Row 1 (months) and Row 2 (days)."""
    month_starts = {}
    for cell in ws[1]:
        if cell.value and cell.column >= 3:
            parts = str(cell.value).strip().split()
            if len(parts) == 2:
                try:
                    yr = int(parts[0])
                    mo = MONTH_MAP.get(parts[1])
                    if mo:
                        month_starts[cell.column] = (yr, mo)
                except ValueError:
                    pass

    col_to_date = {}
    current_yr, current_mo = None, None
    for cell in ws[2]:
        col = cell.column
        if col < 3:
            continue
        if col in month_starts:
            current_yr, current_mo = month_starts[col]
        val = cell.value
        if val is None:
            continue
        if isinstance(val, (int, float)):
            day = int(val)
        elif hasattr(val, 'day'):
            day = val.day
        else:
            continue
        if current_yr and current_mo and 1 <= day <= 31:
            try:
                col_to_date[col] = date(current_yr, current_mo, day)
            except ValueError:
                pass
    return col_to_date

print('✅ Helper functions ready')

✅ Helper functions ready


In [6]:
# ── Load Excel and extract bookings ────────────────────────────────────────────
print(f'Loading {INPUT_FILE}...')
wb = openpyxl.load_workbook(INPUT_FILE, read_only=False)
ws = wb.worksheets[SHEET_INDEX]
print(f'Sheet: {ws.title} | Rows: {ws.max_row} | Cols: {ws.max_column}')

col_to_date = build_date_map(ws)
dates = sorted(col_to_date.values())
print(f'Date range: {dates[0]} → {dates[-1]} ({len(col_to_date)} columns)')

all_bookings = []
warnings = []
skipped_floors = defaultdict(int)
reactor_count = 0
sorted_cols = sorted(col_to_date.items())

for row in ws.iter_rows(min_row=3, max_row=ws.max_row, values_only=False):
    floor_cell = row[0]
    vessel_cell = row[1]

    floor = str(floor_cell.value).strip() if floor_cell.value else None
    vessel = str(vessel_cell.value).strip() if vessel_cell.value else None

    if not floor or not vessel:
        continue
    if floor not in VALID_FLOORS:
        skipped_floors[floor] += 1
        continue

    reactor_id = normalise(floor, vessel)
    if reactor_id not in VALID_IDS:
        warnings.append(f'Unknown reactor_id skipped: {reactor_id}')
        continue
    reactor_count += 1

    current_project = None
    current_user = None
    booking_start = None
    prev_date = None

    for i, (col, d) in enumerate(sorted_cols):
        cell = row[col - 1]  # row is 0-indexed
        raw  = cell.value
        val  = str(raw).strip() if raw is not None else ''

        is_skip = val in SKIP_VALUES
        is_cont = val.startswith(CONT_PREFIX) and len(val) > 1
        is_proj = val and not is_cont and not is_skip

        if is_proj:
            # Close any open booking first
            if current_project and booking_start:
                all_bookings.append({
                    'reactor_id': reactor_id, 'project': current_project,
                    'user': current_user, 'start': str(booking_start), 'end': str(prev_date)
                })
            # Peek next cell to determine real start date
            # Convention: label cell = prep day (D00 follows) OR label IS Day 0 (D01 follows due to back-to-back overlap)
            next_val = ''
            if i + 1 < len(sorted_cols):
                next_col = sorted_cols[i + 1][0]
                next_raw = row[next_col - 1].value
                next_val = str(next_raw).strip() if next_raw is not None else ''
            if next_val == 'D00':
                # Label is prep marker → real start is next day
                current_project = val
                current_user    = extract_user(cell)
                booking_start   = sorted_cols[i + 1][1]
            elif next_val == 'D01':
                # Label cell IS Day 0 (back-to-back overlap convention)
                current_project = val
                current_user    = extract_user(cell)
                booking_start   = d
            else:
                # Ambiguous: not followed by D00 or D01 → likely data entry error, skip
                warnings.append(f'{reactor_id}: project "{val}" at {d} not followed by D00/D01 '
                                f'(next cell: "{next_val}") — skipped as likely error')
                current_project = None
                current_user    = None
                booking_start   = None

        elif is_skip and current_project:
            all_bookings.append({
                'reactor_id': reactor_id, 'project': current_project,
                'user': current_user, 'start': str(booking_start), 'end': str(prev_date)
            })
            current_project = None
            current_user    = None
            booking_start   = None

        prev_date = d

    if current_project and booking_start:
        all_bookings.append({
            'reactor_id': reactor_id, 'project': current_project,
            'user': current_user, 'start': str(booking_start), 'end': str(prev_date)
        })

# Validate dates
valid_bookings = []
for b in all_bookings:
    try:
        s = date.fromisoformat(b['start'])
        e = date.fromisoformat(b['end'])
        if e < s:
            warnings.append(f'{b["reactor_id"]} | {b["project"]} | end < start — skipped')
        else:
            valid_bookings.append(b)
    except Exception:
        warnings.append(f'{b["reactor_id"]} | {b["project"]} | invalid date — skipped')

print(f'\n✅ Reactors processed : {reactor_count}')
print(f'✅ Bookings extracted : {len(valid_bookings)}')
if skipped_floors:
    for f, c in skipped_floors.items():
        print(f'⚠️ Skipped floor: {repr(f)} ({c} rows)')
if warnings:
    print(f'\n⚠️ Warnings ({len(warnings)}):')
    for w in warnings:
        print(f' {w}')

Loading C:/Users/chan85.kim/Desktop/Jupyter Lab/2026/배양기 운용현황판_streamlit/260514_Bioreactor_현황판.xlsx...
Sheet: 배양기 일정 (2024) | Rows: 168 | Cols: 138
Date range: 2026-04-01 → 2026-07-31 (136 columns)

✅ Reactors processed : 166
✅ Bookings extracted : 1039

⚠️ Warnings (93):
 6층-2L-06: project "누전" at 2026-05-01 not followed by D00/D01 (next cell: "") — skipped as likely error
 6층-2L-09: project "SB43" at 2026-04-07 not followed by D00/D01 (next cell: "D08") — skipped as likely error
 6층-2L-13: project "KLA study" at 2026-04-20 not followed by D00/D01 (next cell: "KLA study") — skipped as likely error
 6층-2L-13: project "KLA study" at 2026-04-21 not followed by D00/D01 (next cell: "") — skipped as likely error
 6층-2L-14: project "KLA study" at 2026-04-20 not followed by D00/D01 (next cell: "KLA study") — skipped as likely error
 6층-2L-14: project "KLA study" at 2026-04-21 not followed by D00/D01 (next cell: "SB37") — skipped as likely error
 6층-2L-18: project "Agitation 점검 필요" at 2026-04-

In [7]:
# ── Preview first 10 bookings ──────────────────────────────────────────────────
import pandas as pd
df_preview = pd.DataFrame(valid_bookings[:10])
print('\n=== Preview (first 10 bookings) ===')
print(df_preview.to_string(index=False))


=== Preview (first 10 bookings) ===
reactor_id project      user      start        end
  6층-5L-01    E313   Unknown 2026-04-11 2026-04-19
  6층-5L-01    SB33   MS1 손은민 2026-04-30 2026-05-07
  6층-5L-01     SB3  배양그룹 김지환 2026-05-09 2026-05-19
  6층-5L-01    E313   Unknown 2026-05-23 2026-05-28
  6층-5L-01    SB39 배양 선행표 전문 2026-05-30 2026-06-04
  6층-5L-01    SB28  배양그룹 김형은 2026-06-09 2026-06-16
  6층-5L-01    E304 배양 임경택 전문 2026-06-25 2026-06-30
  6층-5L-02    E313   Unknown 2026-04-14 2026-04-19
  6층-5L-02    SB36       장혜진 2026-04-25 2026-05-03
  6층-5L-02   E313E   Unknown 2026-05-07 2026-05-11


In [8]:
# ── Write gantt_import.csv ─────────────────────────────────────────────────────
fields = ['booking_id', 'reactor_id', 'project', 'user', 'stage', 'method', 'start', 'end']

with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    for i, b in enumerate(valid_bookings, 1):
        writer.writerow({
            'booking_id': f'GANTT_{i:04d}',
            'reactor_id': b['reactor_id'],
            'project': b['project'],
            'user': b['user'],
            'stage': 'N',
            'method': 'Fed-batch',
            'start': b['start'],
            'end': b['end'],
        })

print(f'\n✅ Saved: {OUTPUT_FILE}')
print(f' Total bookings : {len(valid_bookings)}')
print(f' → Upload to Bioreactor App → Admin → Section 3')


✅ Saved: C:/Users/chan85.kim/Desktop/Jupyter Lab/2026/배양기 운용현황판_streamlit/extracted.csv
 Total bookings : 1039
 → Upload to Bioreactor App → Admin → Section 3
